# Примеры применения SiLK Suite через PySiLK

По мотивам главы 9 «The SiLK Suite» из книги:  
> Michael Collins. *Network Security Through Data Analysis: From Data to Action*. 2nd ed. O'Reilly Media, 2017. ISBN 978-1-491-96284-8.

**Источники данных для примеров:**
- CERT SiLK Repository (учебные данные Cyber Defense Exercise): https://tools.netsa.cert.org/silk/repository.html
- Документация PySiLK: https://tools.netsa.cert.org/silk/pySiLK.html
- Исходный код SiLK (включает PySiLK): https://tools.netsa.cert.org/silk/download.html

## Установка и настройка окружения

PySiLK поставляется вместе с пакетом SiLK и доступен после его сборки.  
Переменные окружения, которые должны быть установлены:
- `SILK_DATA_ROOTDIR` — путь к хранилищу flow-данных  
- `SILK_CONFIG_FILE` — путь к конфигурационному файлу `silk.conf`

In [ ]:
import os
import sys
from datetime import datetime, timedelta
from collections import defaultdict

# Базовые зависимости для анализа и визуализации
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Основной модуль PySiLK
# Требует установленного SiLK с поддержкой Python-биндингов
import silk
from silk import (
    silkfile_open, READ, WRITE, APPEND,
    IPSet, IPWildcard, IPAddr,
    silk_is_ipv6, silk_version
)
import silk.site

print(f"PySiLK version: {silk_version()}")
print(f"IPv6 support:   {silk_is_ipv6()}")

# Укажите путь к вашему хранилищу данных
# Для учебных данных CERT CDX:
# os.environ['SILK_DATA_ROOTDIR'] = '/path/to/silk/data'
# os.environ['SILK_CONFIG_FILE']  = '/path/to/silk.conf'

# Инициализация site (читает SILK_CONFIG_FILE)
silk.site.init_site()

---
## Пример 1. Просмотр flow-записей (`rwcut`)

**Задача:** прочитать SiLK-файл и вывести записи в читаемом виде — аналог `rwcut --fields=...`

Оригинальная команда:
```bash
rwcut --fields=sIP,dIP,sPort,dPort,proto,bytes,packets,duration data.rw
rwcut --fields=sIP,dIP,sPort,dPort,proto --num-recs=10 data.rw
rwcut --fields=sIP,dIP,sPort,dPort,proto --no-titles --delimited=, data.rw
```

**Источник данных:** файл `data.rw` из репозитория CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# Путь к учебному файлу данных (измените на свой)
DATA_FILE = 'data.rw'

# --- Шаг 1: открыть файл и считать записи ---
records = []
with silkfile_open(DATA_FILE, READ) as f:
    for rec in f:
        records.append({
            'sIP':      str(rec.sip),
            'dIP':      str(rec.dip),
            'sPort':    rec.sport,
            'dPort':    rec.dport,
            'proto':    rec.protocol,
            'bytes':    rec.bytes,
            'packets':  rec.packets,
            'duration': rec.duration,   # в миллисекундах
            'sTime':    rec.stime,      # объект datetime
        })

df = pd.DataFrame(records)
print(f"Всего записей: {len(df)}")

# --- Шаг 2: вывести первые 10 записей (аналог --num-recs=10) ---
print("\n--- Первые 10 записей ---")
print(df[['sIP', 'dIP', 'sPort', 'dPort', 'proto', 'bytes', 'packets', 'duration']].head(10).to_string(index=False))

# --- Шаг 3: вывести как CSV (аналог --no-titles --delimited=,) ---
print("\n--- CSV-формат (без заголовков) ---")
print(df[['sIP', 'dIP', 'sPort', 'dPort', 'proto']].head(5).to_csv(index=False, header=False))

---
## Пример 2. Фильтрация трафика по порту и протоколу (`rwfilter`)

**Задача:** выделить все TCP-потоки на порт 80 (HTTP) — аналог `rwfilter --proto=6 --dport=80`

Оригинальная команда:
```bash
rwfilter --start-date=2009/02/12 \
         --proto=6 --dport=80 \
         --pass=http_traffic.rw --fail=/dev/null
```

**Источник данных:** учебный набор CERT CDX, дата 2009-02-12  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# --- Шаг 1: задать параметры запроса ---
TARGET_DATE_STR = '2009/02/12'
TARGET_DATE     = datetime.strptime(TARGET_DATE_STR, '%Y/%m/%d')
PROTO_TCP       = 6
PORT_HTTP       = 80
OUTPUT_FILE     = 'http_traffic.rw'

# --- Шаг 2: получить список файлов из репозитория за указанную дату ---
# silk.site.repository_iter возвращает пути к .rw-файлам по параметрам
repo_files = list(silk.site.repository_iter(
    start_date=TARGET_DATE,
    end_date=TARGET_DATE,
    flowtypes=None   # все типы трафика (in, out, inweb, outweb и т.д.)
))
print(f"Файлов в репозитории за {TARGET_DATE_STR}: {len(repo_files)}")

# --- Шаг 3: фильтрация — аналог --proto=6 --dport=80 --pass=... ---
http_records = []
total_read   = 0

with silkfile_open(OUTPUT_FILE, WRITE) as out_f:
    for path in repo_files:
        with silkfile_open(path, READ) as in_f:
            for rec in in_f:
                total_read += 1
                # Условие фильтрации: TCP + порт назначения 80
                if rec.protocol == PROTO_TCP and rec.dport == PORT_HTTP:
                    out_f.write(rec)
                    http_records.append({
                        'sIP':   str(rec.sip),
                        'dIP':   str(rec.dip),
                        'sPort': rec.sport,
                        'dPort': rec.dport,
                        'bytes': rec.bytes,
                        'sTime': rec.stime,
                    })

df_http = pd.DataFrame(http_records)
print(f"Прочитано всего:     {total_read} записей")
print(f"Отфильтровано HTTP:  {len(df_http)} записей")
print(f"Записано в файл:     {OUTPUT_FILE}")
print()

# --- Шаг 4: проверить результат — первые строки ---
print(df_http.head(10).to_string(index=False))

In [ ]:
# Вспомогательная функция для повторного использования фильтрации
# (аналог конвейера rwfilter ... | rwcut ...)

def filter_flows(repo_files, proto=None, dport=None, sip_range=None, dip_range=None):
    """Генератор: возвращает записи, удовлетворяющие условиям фильтра."""
    sip_wc = IPWildcard(sip_range) if sip_range else None
    dip_wc = IPWildcard(dip_range) if dip_range else None

    for path in repo_files:
        with silkfile_open(path, READ) as f:
            for rec in f:
                if proto      is not None and rec.protocol != proto:  continue
                if dport      is not None and rec.dport    != dport:  continue
                if sip_wc     and rec.sip not in sip_wc:              continue
                if dip_wc     and rec.dip not in dip_wc:              continue
                yield rec

# Пример использования — HTTP-трафик через генератор:
count = sum(1 for _ in filter_flows(repo_files, proto=6, dport=80))
print(f"HTTP-потоков (через генератор): {count}")

---
## Пример 3. Подсчёт объёма трафика по времени (`rwcount`)

**Задача:** построить временной ряд байт/потоков для обнаружения аномалий —  
аналог `rwcount --bin-size=3600`.

Оригинальная команда:
```bash
rwfilter --start-date=2009/02/12 --daddress=10.0.0.5 \
         --pass=stdout --fail=/dev/null | \
rwcount --bin-size=3600
```

**Источник данных:** учебный набор CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
from datetime import timezone

# --- Шаг 1: задать параметры ---
BIN_SIZE_SEC  = 3600        # 1 час
TARGET_SERVER = '10.0.0.5'  # IP сервера, трафик к которому анализируем

server_ip = IPAddr(TARGET_SERVER)

# --- Шаг 2: агрегировать записи по временным бинам ---
bins_flows   = defaultdict(int)
bins_bytes   = defaultdict(int)
bins_packets = defaultdict(int)

for path in repo_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            # Фильтр: только трафик к целевому серверу
            if rec.dip != server_ip:
                continue
            # Вычислить ключ бина: время начала потока, округлённое до BIN_SIZE_SEC
            ts  = rec.stime.timestamp()
            bin_key = datetime.fromtimestamp(
                (ts // BIN_SIZE_SEC) * BIN_SIZE_SEC, tz=timezone.utc
            )
            bins_flows[bin_key]   += 1
            bins_bytes[bin_key]   += rec.bytes
            bins_packets[bin_key] += rec.packets

# --- Шаг 3: собрать в DataFrame ---
df_count = pd.DataFrame({
    'datetime': sorted(bins_flows.keys()),
    'Flows':    [bins_flows[k]   for k in sorted(bins_flows)],
    'Bytes':    [bins_bytes[k]   for k in sorted(bins_bytes)],
    'Packets':  [bins_packets[k] for k in sorted(bins_packets)],
})

print(df_count.to_string(index=False))

In [ ]:
# --- Шаг 4: построить временной ряд (как в главе 11 книги) ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].bar(df_count['datetime'], df_count['Bytes'] / 1e6, width=0.03, color='steelblue')
axes[0].set_ylabel('Мегабайты')
axes[0].set_title(f'Трафик к {TARGET_SERVER} по часам')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_count['datetime'], df_count['Flows'], width=0.03, color='darkorange')
axes[1].set_ylabel('Потоков')
axes[1].set_xlabel('Время')
axes[1].grid(axis='y', alpha=0.3)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.tight_layout()
plt.savefig('rwcount_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Шаг 5: базовая статистика для построения аномального детектора ---
print("\nСтатистика по байтам (за все бины):")
print(df_count['Bytes'].describe())

threshold_95 = df_count['Bytes'].quantile(0.95)
print(f"\n95-й перцентиль (порог тревоги): {threshold_95:,.0f} байт")
anomalous = df_count[df_count['Bytes'] > threshold_95]
print(f"Бинов, превышающих порог:         {len(anomalous)}")

---
## Пример 4. IP-множества: создание и использование в фильтрах (`rwset`)

**Задача:** сформировать список подозрительных IP-адресов (сканеров «тёмных адресов») и  
использовать его как фильтр — аналог `rwset`, `rwsetcat`, `rwsettool`.

Оригинальные команды:
```bash
rwfilter ... --daddress=10.99.0.0/16 --pass=stdout | rwset --sip-file=scanners.set
rwsetcat scanners.set
rwsettool --union known_bad.set scanners.set > combined.set
rwfilter ... --sipset=scanners.set --pass=scanner_activity.rw
```

**Источник данных:** учебный набор CERT CDX (тёмные адреса — неиспользуемое адресное пространство)  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# --- Шаг 1: создать IP-множество сканеров по трафику к «тёмным адресам» ---
DARK_NET = '10.99.0.0/16'  # неиспользуемое адресное пространство
dark_wc  = IPWildcard(DARK_NET)

scanners_set = IPSet()

for path in repo_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            if rec.dip in dark_wc:
                scanners_set.add(rec.sip)  # источник -> подозрительный

print(f"Обнаружено уникальных сканеров: {len(scanners_set)}")

# --- Шаг 2: просмотреть содержимое множества (аналог rwsetcat) ---
print("\nПервые 20 адресов из множества:")
for i, ip in enumerate(scanners_set):
    print(f"  {ip}")
    if i >= 19:
        break

# --- Шаг 3: сохранить множество в файл ---
scanners_set.save('scanners.set')
print("\nМножество сохранено: scanners.set")

In [ ]:
# --- Шаг 4: загрузить существующее множество и объединить (аналог rwsettool --union) ---
# Предположим, у нас есть файл known_bad.set с известными вредоносными адресами
try:
    known_bad_set = IPSet.load('known_bad.set')
    print(f"Загружено known_bad.set: {len(known_bad_set)} адресов")
except FileNotFoundError:
    # Для демонстрации создадим пустое множество
    known_bad_set = IPSet()
    known_bad_set.add(IPAddr('10.1.2.3'))
    known_bad_set.add(IPAddr('10.4.5.6'))
    print("Файл known_bad.set не найден — используем демо-множество")

# Объединение: scanners ∪ known_bad
combined_set = scanners_set | known_bad_set
combined_set.save('combined.set')
print(f"\nОбъединённое множество: {len(combined_set)} адресов → combined.set")

# Пересечение и разность — для проверки
intersection = scanners_set & known_bad_set
print(f"Пересечение (новые совпадения): {len(intersection)} адресов")

In [ ]:
# --- Шаг 5: использовать IP-множество как фильтр (аналог --sipset=scanners.set) ---
loaded_set = IPSet.load('scanners.set')

scanner_activity = []
with silkfile_open('scanner_activity.rw', WRITE) as out_f:
    for path in repo_files:
        with silkfile_open(path, READ) as in_f:
            for rec in in_f:
                if rec.sip in loaded_set:  # источник входит в множество сканеров
                    out_f.write(rec)
                    scanner_activity.append({
                        'sIP':   str(rec.sip),
                        'dIP':   str(rec.dip),
                        'dPort': rec.dport,
                        'proto': rec.protocol,
                        'bytes': rec.bytes,
                    })

df_scanners = pd.DataFrame(scanner_activity)
print(f"Потоков от сканеров: {len(df_scanners)}")
if not df_scanners.empty:
    print("\nТоп-10 целевых портов:")
    print(df_scanners['dPort'].value_counts().head(10))

---
## Пример 5. Статистика уникальных значений (`rwuniq`)

**Задача:** найти наиболее активные источники трафика по объёму переданных байт —  
аналог `rwuniq --fields=sIP --values=bytes --sort-output --reverse`.

Оригинальная команда:
```bash
rwfilter --start-date=2009/02/12 --type=out,outweb \
         --pass=stdout --fail=/dev/null | \
rwuniq --fields=sIP --values=bytes --sort-output --reverse
```

**Источник данных:** учебный набор CERT CDX  
https://tools.netsa.cert.org/silk/repository.html

In [ ]:
# --- Шаг 1: получить исходящий трафик (типы out/outweb) ---
outbound_files = list(silk.site.repository_iter(
    start_date=TARGET_DATE,
    end_date=TARGET_DATE,
    flowtypes=['all:out', 'all:outweb']  # зависит от имён классов в silk.conf
))

# --- Шаг 2: агрегировать по sIP ---
agg_bytes    = defaultdict(int)
agg_packets  = defaultdict(int)
agg_flows    = defaultdict(int)
agg_dip_set  = defaultdict(set)

for path in outbound_files:
    with silkfile_open(path, READ) as f:
        for rec in f:
            sip = str(rec.sip)
            agg_bytes[sip]   += rec.bytes
            agg_packets[sip] += rec.packets
            agg_flows[sip]   += 1
            agg_dip_set[sip].add(str(rec.dip))

# --- Шаг 3: собрать результат и отсортировать по убыванию байт ---
df_uniq = pd.DataFrame({
    'sIP':          list(agg_bytes.keys()),
    'Bytes':        [agg_bytes[ip]         for ip in agg_bytes],
    'Packets':      [agg_packets[ip]       for ip in agg_bytes],
    'Flows':        [agg_flows[ip]         for ip in agg_bytes],
    'dIP-Distinct': [len(agg_dip_set[ip])  for ip in agg_bytes],
}).sort_values('Bytes', ascending=False).reset_index(drop=True)

print("Топ-20 источников исходящего трафика (по байтам):")
print(df_uniq.head(20).to_string(index=False))

In [ ]:
# --- Шаг 4: визуализация — кандидаты на утечку данных ---
top20 = df_uniq.head(20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top20['sIP'][::-1], top20['Bytes'][::-1] / 1e6, color='crimson', alpha=0.8)
ax.set_xlabel('Мегабайты')
ax.set_title('Топ-20 источников исходящего трафика')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('rwuniq_toptalkers.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Шаг 5: флаг потенциальной утечки — много байт + много уникальных адресатов ---
BYTES_THRESHOLD  = df_uniq['Bytes'].quantile(0.95)
DIP_THRESHOLD    = df_uniq['dIP-Distinct'].quantile(0.90)

suspects = df_uniq[
    (df_uniq['Bytes']        > BYTES_THRESHOLD) &
    (df_uniq['dIP-Distinct'] > DIP_THRESHOLD)
]
print(f"\nПодозрительных источников (высокий трафик + много адресатов): {len(suspects)}")
print(suspects.to_string(index=False))

---
## Пример 6. Конвертация pcap → SiLK через YAF и анонимизация IP

**Задача:** получить SiLK-данные из pcap-файла и при необходимости анонимизировать IP —  
аналог связки `yaf | rwipfix2silk` и утилиты `rwrandomizeip`.

Оригинальные команды:
```bash
yaf --in capture.pcap --out - | rwipfix2silk --silk-output=flows.rw
rwfileinfo flows.rw
rwrandomizeip --seed=12345 flows.rw anonymized.rw
```

**Источники:**  
- YAF (Yet Another Flowmeter): https://tools.netsa.cert.org/yaf/index.html  
- PySiLK IPAddr: https://tools.netsa.cert.org/silk/pySiLK-notes.html

In [ ]:
import subprocess

PCAP_FILE   = 'capture.pcap'
IPFIX_FILE  = 'capture.ipfix'
SILK_OUTPUT = 'flows.rw'

# --- Шаг 1: запустить yaf для генерации IPFIX из pcap ---
# yaf должен быть установлен отдельно: https://tools.netsa.cert.org/yaf/
yaf_cmd = [
    'yaf',
    '--in',  PCAP_FILE,
    '--out', IPFIX_FILE,
    '--silk',  # включить режим совместимости с SiLK
]
result = subprocess.run(yaf_cmd, capture_output=True, text=True)
if result.returncode == 0:
    print(f"yaf: IPFIX-файл создан → {IPFIX_FILE}")
else:
    print(f"yaf завершился с ошибкой:\n{result.stderr}")

# --- Шаг 2: конвертировать IPFIX → SiLK через rwipfix2silk ---
conv_cmd = [
    'rwipfix2silk',
    f'--silk-output={SILK_OUTPUT}',
    IPFIX_FILE,
]
result2 = subprocess.run(conv_cmd, capture_output=True, text=True)
if result2.returncode == 0:
    print(f"rwipfix2silk: SiLK-файл создан → {SILK_OUTPUT}")
else:
    print(f"rwipfix2silk завершился с ошибкой:\n{result2.stderr}")

In [ ]:
# --- Шаг 3: просмотреть метаданные файла (аналог rwfileinfo) ---
with silkfile_open(SILK_OUTPUT, READ) as f:
    print(f"Формат:       {f.format}")
    print(f"Версия:       {f.version}")
    print(f"Тип данных:   {f.type}")
    print(f"Сенсор:       {f.sensor}")

# Подсчёт записей и краткая сводка
rows = []
with silkfile_open(SILK_OUTPUT, READ) as f:
    for rec in f:
        rows.append({'sIP': str(rec.sip), 'dIP': str(rec.dip),
                     'proto': rec.protocol, 'bytes': rec.bytes})

df_flows = pd.DataFrame(rows)
print(f"\nЗаписей в файле: {len(df_flows)}")
print(df_flows.head(10).to_string(index=False))

In [ ]:
import random
import hashlib

# --- Шаг 4: анонимизация IP-адресов (аналог rwrandomizeip --seed=12345) ---
# PySiLK не предоставляет rwrandomizeip напрямую;
# реализуем детерминированное псевдослучайное переименование через хэш

ANON_SEED   = 12345
ANON_OUTPUT = 'anonymized.rw'

def anonymize_ip(ip_addr, seed=ANON_SEED):
    """Детерминированная анонимизация: заменяет IP псевдослучайным адресом.
    Тот же IP всегда получает один и тот же псевдоним (для сохранения связности).
    """
    h = hashlib.md5(f"{seed}:{ip_addr}".encode()).digest()
    # Используем только первые 4 байта для IPv4
    return IPAddr(f"{h[0]}.{h[1]}.{h[2]}.{h[3]}")

with silkfile_open(ANON_OUTPUT, WRITE) as out_f:
    with silkfile_open(SILK_OUTPUT, READ) as in_f:
        for rec in in_f:
            rec.sip = anonymize_ip(rec.sip)
            rec.dip = anonymize_ip(rec.dip)
            out_f.write(rec)

print(f"Анонимизированный файл сохранён: {ANON_OUTPUT}")

# Проверка — убеждаемся, что адреса изменились
orig_sample  = []
anon_sample  = []

with silkfile_open(SILK_OUTPUT, READ) as f:
    for i, rec in enumerate(f):
        orig_sample.append(str(rec.sip))
        if i >= 4: break

with silkfile_open(ANON_OUTPUT, READ) as f:
    for i, rec in enumerate(f):
        anon_sample.append(str(rec.sip))
        if i >= 4: break

print("\nОригинальные sIP → Анонимизированные sIP:")
for orig, anon in zip(orig_sample, anon_sample):
    print(f"  {orig:20s} → {anon}")

---
## Ссылки на источники

| Ресурс | URL |
|--------|-----|
| Инструментарий SiLK (Carnegie Mellon CERT) | https://tools.netsa.cert.org/silk/ |
| Документация PySiLK | https://tools.netsa.cert.org/silk/pySiLK.html |
| Репозиторий учебных данных SiLK (CDX) | https://tools.netsa.cert.org/silk/repository.html |
| YAF (Yet Another Flowmeter) | https://tools.netsa.cert.org/yaf/index.html |
| Примеры кода к книге | https://github.com/mpcollins/nsda_examples |
| Страница книги (errata + доп. материалы) | http://bit.ly/nstda2e |